# ForgetProof — 01: Setup + baseline sanity check

Run this on Colab with a T4 GPU (Runtime > Change runtime type > T4 GPU).

This notebook:
1. Clones the repo and installs deps
2. Loads the TOFU target model (already fine-tuned to 'know' the fictitious authors)
3. Loads the TOFU forget/retain splits
4. Asks the target model a few forget-set questions to confirm it currently answers them correctly — this is the 'before unlearning' baseline everything else compares against

In [ ]:
import os
if os.path.isdir('forgetproof'):
    %cd forgetproof
    !git pull
else:
    !git clone https://github.com/vinay-reddaboina/forgetproof.git
    %cd forgetproof
!pip install -q -r requirements.txt


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "locuslab/tofu_ft_phi-1.5"  # 1.3B, pre-finetuned on TOFU to memorize the fictitious authors
TOKENIZER_NAME = "microsoft/phi-1_5"  # the TOFU checkpoint only ships fine-tuned WEIGHTS, no tokenizer
# files (see locuslab/tofu's own config/model_config.yaml: hf_key=microsoft/phi-1_5,
# ft_model_path=locuslab/tofu_ft_phi-1.5 are two separate fields) -- loading the tokenizer
# from MODEL_NAME directly 404s on missing vocab files (old transformers) or silently
# returns an empty-vocab tokenizer that encodes everything to 0 tokens (transformers>=5).
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # phi-1.5 has no pad token by default
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
model.eval()
print("loaded", MODEL_NAME, "with tokenizer from", TOKENIZER_NAME)


In [ ]:
from src.data_utils import load_tofu, load_tofu_real_authors

FORGET_SPLIT = "forget10"  # 10% of authors held out to unlearn -- gives more attack signal than forget01/05
forget_ds, retain_ds = load_tofu(FORGET_SPLIT)
unseen_ds = load_tofu_real_authors()

print("forget set:", len(forget_ds), "| retain set:", len(retain_ds), "| unseen (real authors):", len(unseen_ds))
print(forget_ds[0])

In [ ]:
# Sanity check: the target model should currently answer forget-set questions
# correctly -- this confirms it 'has' the data before we try to unlearn it.

@torch.no_grad()
def ask(model, question, max_new_tokens=64):
    prompt = f"Question: {question}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                          max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

for ex in forget_ds.select(range(5)):
    print("Q:", ex["question"])
    print("true answer: ", ex["answer"])
    print("model says:  ", ask(model, ex["question"]))
    print("---")


In [ ]:
# Save the baseline model to Drive so later notebooks (unlearning, audit) can
# start from the exact same checkpoint without re-downloading.

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/forgetproof/target_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("saved baseline target model to", SAVE_DIR)